# 00. 설정과 데모

레포를 받고, 테스트를 돌리고, 합성 샘플로 사진 → 안내 문장 → 음성 한 바퀴를 확인한다. 마지막에 웹 데모를 띄운다.

- 처음엔 OCR 이 가짜(`dummy`)라서 `data/samples` 의 합성 사진에서만 글자가 나온다.
- 진짜 OCR 은 `PACK_OCR=easyocr` 처럼 환경 변수로 바꾼다 (docs/colab_guide.md).

In [ ]:
# 레포 받기 (이미 있으면 최신으로) + 기본 설치
import os
REPO = "/content/AI_tts_web"
if not os.path.exists(REPO):
    !git clone -q https://github.com/DL-2026F-Team05/AI_tts_web.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt -r requirements-dev.txt
!git log --oneline -3

## 테스트
`xfailed` 는 아직 구현 안 된 명세(정상). `failed` 가 있으면 담당자에게 알린다.

In [ ]:
!python -m pytest -q

## 샘플로 한 바퀴
알레르기로 우유, 땅콩을 등록했다고 치고 돌린다.

In [ ]:
from packreader.config import Config
from packreader.pipeline import Pipeline
from packreader.schema import UserProfile

pipe = Pipeline(Config.from_env(tts_engines=()))  # 여기서는 음성 없이 글자만
profile = UserProfile(allergens=["우유", "땅콩"])
for sid in ["S001", "S002", "S003", "S004", "S005", "S006"]:
    r = pipe.analyze(f"data/samples/{sid}.jpg", profile=profile, with_audio=False)
    print(f"[{sid}] 위험도={r.risk.level}")
    print("  ", r.speech.summary)

## 인식 결과 그려 보기

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

COLORS = {"product_name": "purple", "expiry": "red", "allergen": "orange", "ingredients": "teal",
          "nutrition": "blue", "caution": "gray", "other": "lightgray", None: "black"}

def show(path, result, width=700):
    img = Image.open(path).convert("RGB")
    d = ImageDraw.Draw(img)
    for b in result.boxes:
        if b.poly:
            d.polygon([tuple(p) for p in b.poly], outline=COLORS.get(b.region, "black"), width=3)
    img.thumbnail((width, width))
    display(img)

r = pipe.analyze("data/samples/S001.jpg", profile=profile, with_audio=False)
show("data/samples/S001.jpg", r)
print(r.fields.model_dump_json(indent=1)[:1500])

## 음성 들어 보기 (선택)
인터넷 음성 엔진(edge-tts, gTTS)으로 요약 문장을 읽는다. MeloTTS 같은 로컬 모델 비교는 `01_tts_compare.ipynb`.

In [ ]:
!pip install -q -r requirements-tts.txt

In [ ]:
from IPython.display import Audio, display
from packreader.speech.tts import TTSManager

tts = TTSManager.from_names(["edge", "gtts"], cache_dir="/content/tts_cache")
r = pipe.analyze("data/samples/S001.jpg", profile=profile, with_audio=False)
clip = tts.synthesize(r.speech.summary)
print("엔진:", clip.engine if clip else "없음", "| 실패:", tts.last_errors)
if clip:
    display(Audio(filename=str(tts.last_path)))

## 진짜 OCR(EasyOCR)로 확인
- 위 결과는 가짜 OCR(정답)이라 다 맞는다. 실제로는 이렇게 틀리게 읽는다는 걸 여기서 본다
- 런타임을 **T4 GPU** 로 바꾸고 돌리는 걸 권한다. CPU 면 한 장에 15~20초 걸린다
- 실제 OCR 로 확인한 문제와 대응: `docs/ocr_findings.md`

In [ ]:
!pip install -q easyocr
import os
os.environ["PACK_OCR"] = "easyocr"
real = Pipeline(Config.from_env(tts_engines=()))
for sid in ["S001", "S003", "S005"]:
    r = real.analyze(f"data/samples/{sid}.jpg", profile=profile, with_audio=False)
    print(f"=== {sid} ({r.timing_ms.get('total', 0) / 1000:.1f}초, 보정={r.preprocess_ops})")
    for b in r.boxes:
        print(f"  {b.conf:.2f} | {b.text}")
    print("  →", r.speech.summary)
os.environ["PACK_OCR"] = "dummy"  # 아래 웹 데모는 다시 가짜 OCR 로 (바꾸려면 이 줄을 지운다)

## 웹 데모
- 첫 번째 주소: 이 PC 브라우저에서만 열린다
- 두 번째 주소(cloudflared): 폰에서도 열린다. HTTPS 라서 폰 카메라도 된다

In [ ]:
import os, subprocess, sys, time
os.environ.setdefault("PACK_TTS", "melo,edge,gtts")  # 되는 엔진을 앞에서부터 쓰고, 다 안 되면 브라우저 음성
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!tail -n 3 /content/server.log
from google.colab.output import eval_js
print("PC 브라우저용 주소:", eval_js("google.colab.kernel.proxyPort(8000)"))

In [ ]:
# 폰으로 열 주소 (cloudflared 임시 터널)
import re
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("폰에서 열 주소:", url or "아직 안 나옴. /content/tunnel.log 확인")

In [ ]:
# 끄기
for name in ("tunnel", "server"):
    if name in globals():
        globals()[name].terminate()
print("종료")